# LSTMD Workload Forecasting Benchmark

This Kaggle notebook evaluates LSTMD on all 12 official workload traces and the single/multiple training, CPU, memory, and joint CPU+memory settings. It retains the paper's 288-step input, 2-step horizon, saved hyperparameters, and batch sizes. Each fit is capped at **500 epochs**, with **patience 50** and **one seed**; early stopping may end a fit sooner. This is 25% of the official epoch cap and 10% of its ten repetitions, so report it as a reduced reproduction.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = "https://github.com/grass-overflow/ML_Proj.git"
repo_directory = Path("/kaggle/working/ML_Proj")
if (repo_directory / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_directory), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_directory)], check=True)
os.chdir(repo_directory)
missing = [name for name in ("requirements-cpu.txt", "run_experiments.py") if not Path(name).is_file()]
if missing:
    raise RuntimeError("Repository checkout is missing: " + ", ".join(missing))
print("Repository ready at", Path.cwd())

### Install Kaggle-Compatible Dependencies

Kaggle's preinstalled JAX conflicts with TensorFlow 2.16's pinned `ml_dtypes`; remove JAX before installing project requirements.

In [ ]:
%pip uninstall -y jax jaxlib
%pip install -q -r requirements-cpu.txt

### Run LSTMD Across All Traces

This loops over all 12 targets. Each target runs three prediction tasks under single- and all-12-trace training: up to 72 fits. Every fit has a 500-epoch cap, patience 50, one seed, and the saved batch size. Enable Kaggle GPU acceleration before running.

In [ ]:
import os
import subprocess

os.environ["WORKLOAD_USE_GPU"] = "1"
model_name = "LSTMD"
datasets = ["ali18", "ali20_c", "ali20_g", "gc11", "gc19_a", "gc19_b", "gc19_c", "gc19_d", "gc19_e", "gc19_f", "gc19_g", "gc19_h"]
common = ["--models", model_name, "--runs", "1", "--epochs", "500", "--patience", "50", "--paper-faithful", "--mc-samples", "100", "--inference-repeats", "30", "--no-plots"]
for index, dataset in enumerate(datasets, 1):
    print(f"[{index}/{len(datasets)}] LSTMD benchmark: {dataset}", flush=True)
    subprocess.run(["python", "run_experiments.py", "--mode", "model_comparison", "--datasets", dataset] + common, check=True)

from util.experiment_plots import generate_plots
generate_plots()

### Paper-Style Metrics and Visuals

The runner saves prediction intervals, point/quantile errors, service-level calibration, TPR/SR, runtime, uncertainty, and dataset-wise figures. This cell adds all-trace workload distributions, compares LSTMD results with the paper's Table 1, displays the figures, and packages results.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

layers = ["288-step input", "1D convolution", "LSTM + dense", "Deterministic dense", "Gaussian distribution"]
fig, axis = plt.subplots(figsize=(6, 7))
axis.set_xlim(0, 1)
axis.set_ylim(0, len(layers) + 0.5)
axis.axis("off")
for index, label in enumerate(layers):
    y = len(layers) - index - 0.5
    axis.add_patch(FancyBboxPatch((0.12, y - 0.22), 0.76, 0.42, boxstyle="round,pad=0.02", facecolor="#e9f1f5", edgecolor="#315b6d"))
    axis.text(0.5, y, label, ha="center", va="center")
    if index < len(layers) - 1:
        axis.annotate("", (0.5, y - 0.31), (0.5, y - 0.08), arrowprops={"arrowstyle": "->"})
axis.set_title("LSTMD architecture")
fig.tight_layout()
fig.savefig("results/plots/lstmd_architecture.png", dpi=150)
display(fig)
plt.close(fig)

fig, axis = plt.subplots(figsize=(10, 3))
axis.axis("off")
workflow = ["Historical workload", "Predictive model", "Resource manager", "Cloud resources"]
positions = [0.12, 0.38, 0.64, 0.88]
for x, label in zip(positions, workflow):
    axis.text(x, 0.55, label, ha="center", va="center", bbox={"boxstyle": "round,pad=0.45", "facecolor": "#e9f1f5", "edgecolor": "#315b6d"})
for left, right in zip(positions, positions[1:]):
    axis.annotate("", (right - 0.08, 0.55), (left + 0.08, 0.55), arrowprops={"arrowstyle": "->"})
axis.annotate("updated history", (0.12, 0.2), (0.88, 0.2), arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=-0.25"}, ha="center")
axis.set_title("Cloud workload prediction workflow")
fig.tight_layout()
fig.savefig("results/plots/workload_prediction_workflow.png", dpi=150)
display(fig)
plt.close(fig)

In [ ]:
from glob import glob
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, Image, display

paper = {
    "S-U-LSTMD": {"avgcpu": (0.0040, 0.0455), "avgmem": (0.0041, 0.0420)},
    "S-B-LSTMD": {"avgcpu": (0.0061, 0.0556), "avgmem": (0.0088, 0.0631)},
    "M-U-LSTMD": {"avgcpu": (0.0041, 0.0464), "avgmem": (0.0044, 0.0439)},
    "M-B-LSTMD": {"avgcpu": (0.0046, 0.0446), "avgmem": (0.0046, 0.0446)},
}
raw = pd.read_csv("results/raw/model_metrics.csv")
actual = raw.groupby(["model", "resource"])[["MSE", "MAE"]].mean().reset_index()
rows = []
for label, resources in paper.items():
    for resource, (paper_mse, paper_mae) in resources.items():
        match = actual[(actual.model == label) & (actual.resource == resource)]
        rows.append({"model": label, "resource": resource, "paper_MSE": paper_mse, "run_MSE": float(match.MSE.iloc[0]) if len(match) else None, "paper_MAE": paper_mae, "run_MAE": float(match.MAE.iloc[0]) if len(match) else None})
comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv("results/metrics/lstmd_vs_paper_table1.csv", index=False)

datasets = ["ali18", "ali20_c", "ali20_g", "gc11", "gc19_a", "gc19_b", "gc19_c", "gc19_d", "gc19_e", "gc19_f", "gc19_g", "gc19_h"]
fig, axes = plt.subplots(6, 4, figsize=(16, 20))
for row, dataset in enumerate(datasets):
    frame = pd.read_csv(Path("saved_data/preprocessed") / f"{dataset}.csv")
    for col, resource in enumerate(("avgcpu", "avgmem")):
        axis = axes[row // 2, (row % 2) * 2 + col]
        axis.hist(frame[resource].dropna(), bins=40, color="#287c8e" if col == 0 else "#e07a5f", alpha=0.85)
        axis.set_title(f"{dataset}: {resource}")
        axis.set_xlabel("Normalized utilization")
        axis.set_ylabel("Frequency")
fig.tight_layout()
fig.savefig("results/plots/lstmd_workload_distributions.png", dpi=150)
display(fig)
plt.close(fig)

plots = sorted(glob("results/plots/*.png"))
print(f"Generated {len(plots)} plots")
for path in plots:
    print(path)
    display(Image(filename=path))
archive = shutil.make_archive("LSTMD_benchmark_results", "zip", "results")
print(archive)
display(FileLink(archive))